> Public repository edition. The analysis uses the included **sample** by default. Full report results used the separate full dataset. Code paths and validation were cleaned for publication; original score formulas, pair split, model settings and evaluation definitions are retained. Outputs are cleared. See [reproducibility notes](../docs/reproducibility.md).

## Import packages and Load the MinMax dataset

In [ ]:
from pathlib import Path
import sys

# Start Jupyter in the repository root or its notebooks/ directory.
project_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "project_config.py").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("Open this notebook from the repository root or notebooks/ folder.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from project_config import get_paths

paths = get_paths()
print("Data mode:", paths.mode)
print("Input:", paths.input_path)
print("Local processed data:", paths.processed_dir)
print("Local evaluation outputs:", paths.evaluation_dir)

# Setting up packages, input file, and output location

import pandas as pd
import numpy as np
from pathlib import Path

# Main input file
input_path = paths.input_path

# Output file
output_dir = paths.processed_dir
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "IdentityMF_MinMax_base_dataset.csv"

In [ ]:
columns = [
    "compound_1", "compound_2", "minS11", "minS22", "maxS12", "delta_prime", "label"
]
# The input is tab-separated and has no header.
if not input_path.is_file():
    raise FileNotFoundError(f"Input missing: {input_path}. See data/README.md.")
df = pd.read_csv(input_path, sep="\t", header=None)
if df.shape[1] != len(columns):
    raise ValueError(f"Expected seven input columns; received {df.shape[1]}.")
df.columns = columns
if df.isna().any().any():
    raise ValueError("Input contains missing values.")
numeric_columns = ["minS11", "minS22", "maxS12", "delta_prime"]
if not np.isfinite(df[numeric_columns].to_numpy()).all():
    raise ValueError("Score columns must contain finite numeric values.")
if not df[numeric_columns].ge(0).all().all() or not df[numeric_columns].le(1).all().all():
    raise ValueError("Similarity components and delta_prime must be in [0, 1].")
if not df["label"].isin([0, 1]).all():
    raise ValueError("Labels must be 0 or 1.")
if not ((df["compound_1"] == df["compound_2"]).astype(int) == df["label"]).all():
    raise ValueError("Labels disagree with exact compound-code identity.")
expected_rows = 20_736 if paths.mode == "sample" else 2_073_600
expected_positive = 144 if paths.mode == "sample" else 14_400
if len(df) != expected_rows or int(df["label"].sum()) != expected_positive:
    raise ValueError(
        f"Unexpected {paths.mode} dataset size/composition: "
        f"{len(df):,} rows, {int(df['label'].sum()):,} positives. "
        f"Expected {expected_rows:,} rows and {expected_positive:,} positives."
    )
df.head()


In [ ]:
# Checking that the file loaded correctly

print("Shape:", df.shape)

print("\nColumn types:")
print(df.dtypes)

print("\nLabel counts:")
print(df["label"].value_counts())

print("\nLabel proportions:")
print(df["label"].value_counts(normalize=True))

## Reconstruct the original min-max index and Create the sign-reversed score *r*

The file contains the transformed min-max score used in the paper, but for calibration we want to reconstruct the original value:

$$
\delta=min(min(S_{11}),min(S_{22}))−max(S_{12})
$$

This original $\delta$ preserves negative values, which are lost in the transformed version.

The original $\delta$ is larger when there is stronger evidence that two compounds are different. Since our calibration target is the probability of same-compound classification, it is more intuitive to define:

$$
r = −\delta
$$

With this definition, larger r generally means more overlap between the two samples, so it should usually correspond to a higher probability of `label = 1`.

In [ ]:
# Computing the original min-max index before transformation and Creating r_overlap = -delta_raw

df["within_min"] = df[["minS11", "minS22"]].min(axis=1)

df["delta_raw"] = df["within_min"] - df["maxS12"]

df["r_overlap"] = -df["delta_raw"]

df[["minS11", "minS22", "within_min", "maxS12", "delta_raw", "r_overlap"]].head()

## Check that our reconstructed score matches the paper’s transformation

The paper’s transformed score is:

$$
\delta' = 1 − max(0,\delta)
$$

We reconstruct this value and compare it with the `delta_prime` column. This is an important sanity check because it confirms that we interpreted the columns correctly.

In [ ]:
df["delta_prime_reconstructed"] = 1 - np.maximum(0, df["delta_raw"])

df["delta_prime_error"] = (
    df["delta_prime"] - df["delta_prime_reconstructed"]
).abs()

max_error = df["delta_prime_error"].max()

print("Maximum reconstruction error:", max_error)

# Source values were rounded to six decimal places.
if np.allclose(df["delta_prime"], df["delta_prime_reconstructed"], rtol=0, atol=2e-6):
    print("Reconstruction check passed within source-rounding tolerance (2e-6).")
else:
    raise ValueError("Stored delta_prime disagrees with the reconstructed score.")


## Create an order-independent compound-pair ID

Later, when we split the dataset into training and testing sets, we should avoid putting repeated versions of the same compound pair in both sets. To help with this, we create a `pair_id`.

The pair ID is order-independent, so `CM1/01` vs `CM1/02` and `CM1/02` vs `CM1/01` will get the same ID.

In [ ]:
df["pair_id"] = df.apply(
    lambda row: "__".join(sorted([row["compound_1"], row["compound_2"]])),
    axis=1
)

df[["compound_1", "compound_2", "pair_id"]].head()

In [ ]:
base_columns = [
    "compound_1",
    "compound_2",
    "pair_id",
    "minS11",
    "minS22",
    "within_min",
    "maxS12",
    "delta_raw",
    "r_overlap",
    "delta_prime",
    "delta_prime_reconstructed",
    "label"
]

base_df = df[base_columns].copy()

base_df.head()

In [ ]:
# Saving the processed base dataset

base_df.to_csv(output_path, index=False)

print("Saved file:", output_path)
print("Saved shape:", base_df.shape)

In [ ]:
columns = [
    "compound_1", "compound_2", "minS11", "minS22", "maxS12", "delta_prime", "label"
]
# The input is tab-separated and has no header.
if not input_path.is_file():
    raise FileNotFoundError(f"Input missing: {input_path}. See data/README.md.")
df = pd.read_csv(input_path, sep="\t", header=None)
if df.shape[1] != len(columns):
    raise ValueError(f"Expected seven input columns; received {df.shape[1]}.")
df.columns = columns
if df.isna().any().any():
    raise ValueError("Input contains missing values.")
numeric_columns = ["minS11", "minS22", "maxS12", "delta_prime"]
if not np.isfinite(df[numeric_columns].to_numpy()).all():
    raise ValueError("Score columns must contain finite numeric values.")
if not df[numeric_columns].ge(0).all().all() or not df[numeric_columns].le(1).all().all():
    raise ValueError("Similarity components and delta_prime must be in [0, 1].")
if not df["label"].isin([0, 1]).all():
    raise ValueError("Labels must be 0 or 1.")
if not ((df["compound_1"] == df["compound_2"]).astype(int) == df["label"]).all():
    raise ValueError("Labels disagree with exact compound-code identity.")
expected_rows = 20_736 if paths.mode == "sample" else 2_073_600
expected_positive = 144 if paths.mode == "sample" else 14_400
if len(df) != expected_rows or int(df["label"].sum()) != expected_positive:
    raise ValueError(
        f"Unexpected {paths.mode} dataset size/composition: "
        f"{len(df):,} rows, {int(df['label'].sum()):,} positives. "
        f"Expected {expected_rows:,} rows and {expected_positive:,} positives."
    )
df.head()


## Create a train/test split using unique `pair_id`s

Now we create one row per unique `pair_id`. This smaller table is used only for splitting. We keep the label so that the train/test split can preserve the ratio of same-compound and different-compound pairs

In [ ]:
from sklearn.model_selection import train_test_split

# Check that each pair_id has only one label
pair_label_counts = base_df.groupby("pair_id")["label"].nunique()

print("Maximum number of labels within a pair_id:", pair_label_counts.max())

if pair_label_counts.max() != 1:
    raise ValueError("Some pair_id values contain both labels. Check pair_id construction.")

Now we create one row per unique `pair_id`. This smaller table is used only for splitting. We keep the label so that the train/test split can preserve the ratio of same-compound and different-compound pairs.

In [ ]:
pair_df = (
    base_df.groupby("pair_id")
      .agg(
          label=("label", "first"),
          n_rows=("label", "size")
      )
      .reset_index()
)

print("Pair-level dataset shape:", pair_df.shape)
pair_df.head()

We split the unique compound pairs into training and testing sets. The `stratify` option keeps the same-compound and different-compound proportions approximately similar in both sets.

In [ ]:
train_pairs, test_pairs = train_test_split(
    pair_df,
    test_size=0.20,
    random_state=85,
    stratify=pair_df["label"]
)

print("Number of train pairs:", train_pairs.shape[0])
print("Number of test pairs:", test_pairs.shape[0])

Using the selected pair IDs to create the actual row-level training and testing datasets

- All repeated min-max iterations of the same compound pair stay together in either training or testing.

In [ ]:
train_pair_ids = set(train_pairs["pair_id"])
test_pair_ids = set(test_pairs["pair_id"])

train_df = base_df[base_df["pair_id"].isin(train_pair_ids)].copy()
test_df = base_df[base_df["pair_id"].isin(test_pair_ids)].copy()

print("Train dataset shape:", train_df.shape)
print("Test dataset shape:", test_df.shape)

Checking that there is no leakage between training and testing
- A pair_id should appear in only one set
- The overlap count should be zero

In [ ]:
overlap_pairs = set(train_df["pair_id"]).intersection(set(test_df["pair_id"]))

print("Number of overlapping pair_ids:", len(overlap_pairs))

if len(overlap_pairs) == 0:
    print("Check passed: no pair_id appears in both train and test.")
else:
    raise ValueError("A pair_id appears in both train and test.")

Comparing the label distributions in the full dataset, training set, and testing set:

- They should be similar, although not exactly identical.

In [ ]:
def label_summary(data, name):
    counts = data["label"].value_counts().sort_index()
    proportions = data["label"].value_counts(normalize=True).sort_index()
    
    summary = pd.DataFrame({
        "count": counts,
        "proportion": proportions
    })
    
    print(f"\n{name}")
    print(summary)

label_summary(base_df, "Full dataset")
label_summary(train_df, "Train dataset")
label_summary(test_df, "Test dataset")

Checking the split at the pair level
- This confirms how many same-compound and different-compound pairs are in the training and testing sets

In [ ]:
label_summary(pair_df, "Full pair-level dataset")
label_summary(train_pairs, "Train pair-level dataset")
label_summary(test_pairs, "Test pair-level dataset")

### Saving the training and testing datasets.

In [ ]:
output_dir = paths.processed_dir
output_dir.mkdir(parents=True, exist_ok=True)

train_output_path = output_dir / "IdentityMF_MinMax_train.csv"
test_output_path = output_dir / "IdentityMF_MinMax_test.csv"

train_df.to_csv(train_output_path, index=False)
test_df.to_csv(test_output_path, index=False)

print("Train file saved to:", train_output_path)
print("Test file saved to:", test_output_path)